In [1]:
import os
import numpy as np
import pandas as pd

RAW_DATA_PATH = os.path.join("..","data","raw","kaggle_dataset_complete.csv")
PROCESSED_DATA_DIR = os.path.join("..","data","processed")

df_raw = pd.read_csv(RAW_DATA_PATH)
print("Raw data loaded successfully.")
print(f"Initial shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

Raw data loaded successfully.
Initial shape: 7308 rows, 9 columns


In [2]:
print("\n---Integrity Verification---")
missing_total = df_raw.isnull().sum().sum() # null per column + sum of all columns
duplicate_total = df_raw.duplicated().sum()

# physical boundary check
invalid_depth = (df_raw["FloodDepth_m"] < 0.0).sum()
invalid_moisture = ((df_raw["SoilMoisture_pct"] < 0) | (df_raw["SoilMoisture_pct"] > 100)).sum()
invalid_rain = (df_raw["Rainfall_mm"] < 0.0).sum()
invalid_water = (df_raw["WaterLevel_m"] < 0.0).sum()

print(f"Total missing values: {missing_total}")
print(f"Total duplicate rows: {duplicate_total}")
print(f"Negative FloodDepth entries: {invalid_depth}")
print(f"Out-of-range SoilMoisture entries: {invalid_moisture}")
print(f"Negative Rainfall entries: {invalid_rain}")
print(f"Negative WaterLevel entries: {invalid_water}")

assert missing_total == 0, "Warning: Missing values detected that require handling."
assert duplicate_total == 0, "Warning: Duplicate rows detected that require removal."
assert invalid_depth == 0, "Warning: Negative FloodDepth values detected."
assert invalid_moisture == 0, "Warning: SoilMoisture values outside 0–100% detected."
assert invalid_rain == 0, "Warning: Negative Rainfall values detected."
assert invalid_water == 0, "Warning: Negative WaterLevel values detected."

print("Data quality checks passed: Zero nulls and zero duplicates.")


---Integrity Verification---
Total missing values: 0
Total duplicate rows: 0
Negative FloodDepth entries: 0
Out-of-range SoilMoisture entries: 0
Negative Rainfall entries: 0
Negative WaterLevel entries: 0
Data quality checks passed: Zero nulls and zero duplicates.


In [3]:
df = df_raw.copy()

def assign_mgb_risk(depth):
    if depth < 0.5:
        return "Low"
    elif depth <= 1.0:
        return "Moderate"
    else:
        return "High"

if "RiskLevel" not in df.columns:
    df["RiskLevel"] = df["FloodDepth_m"].apply(assign_mgb_risk)
    print("RiskLevel column generated using MGB criteria.")
else:
    derived = df["FloodDepth_m"].apply(assign_mgb_risk)
    mismatches = (df["RiskLevel"] != derived).sum()
    print(f"MGB RiskLevel validation: {mismatches} mismatches found.")

print("\nFinal RiskLevel Distribution:")
display(df["RiskLevel"].value_counts())

MGB RiskLevel validation: 0 mismatches found.

Final RiskLevel Distribution:


RiskLevel
Low         7198
Moderate      63
High          47
Name: count, dtype: int64

In [4]:
# one-hot encoding (convert categorical data (location) into a binary format for machine learning)

df_clean = df_raw.copy()

location_encoded = pd.get_dummies(
    df_clean["Location"], prefix="Location", dtype=int
)

# concatenate horizontally (column)
df_clean = pd.concat([df_clean, location_encoded], axis=1)

print("--- Encoded Location Columns ---")
display(
    df_clean[["Location"] + [col for col in location_encoded.columns]].head()
)

--- Encoded Location Columns ---


,Location,Location_Manila,Location_Marikina,Location_Pasig,Location_Quezon City
0,Quezon City,0,0,0,1
1,Marikina,0,1,0,0
2,Manila,1,0,0,0
3,Pasig,0,0,1,0
4,Quezon City,0,0,0,1


In [5]:
# reorder columns for clear structure: Metadata - Continuous Features - Static/Dummies - Targets
final_columns = [
    "Date",
    "Location",
    "Rainfall_mm",
    "WaterLevel_m",
    "SoilMoisture_pct",
    "Elevation_m",
    "Location_Manila",
    "Location_Marikina",
    "Location_Pasig",
    "Location_Quezon City",
    "FloodOccurrence",
    "FloodDepth_m",
    "RiskLevel"
]

# for exact naming matches
df_clean = df_clean[
    [col for col in final_columns if col in df_clean.columns]
]

print("--- Preprocessed DataFrame Summary ---")
display(df_clean.head())
print(f"Final column count: {df_clean.shape[1]}")

--- Preprocessed DataFrame Summary ---


,Date,Location,Rainfall_mm,WaterLevel_m,SoilMoisture_pct,Elevation_m,Location_Manila,Location_Marikina,Location_Pasig,Location_Quezon City,FloodOccurrence,FloodDepth_m,RiskLevel
0,2016-01-01,Quezon City,12.0,0.5,15.3,43,0,0,0,1,0,0.0,Low
1,2016-01-01,Marikina,10.6,1.8,23.2,15,0,1,0,0,0,0.0,Low
2,2016-01-01,Manila,5.7,0.5,15.6,5,1,0,0,0,0,0.0,Low
3,2016-01-01,Pasig,3.7,0.5,5.0,5,0,0,1,0,0,0.0,Low
4,2016-01-02,Quezon City,3.4,0.5,13.3,43,0,0,0,1,0,0.0,Low


Final column count: 13


In [6]:
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
PROCESSED_FILE_PATH = os.path.join(PROCESSED_DATA_DIR, "cleaned_dataset.csv")

# saving a csv file to /processed 
df_clean.to_csv(PROCESSED_FILE_PATH, index=False)

print("Preprocessed dataset successfully saved.")
print(f"Destination: {PROCESSED_FILE_PATH}")
print(f"Final Exported Shape: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")

df_verify = pd.read_csv(PROCESSED_FILE_PATH)
print("\n--- Verified Export Columns ---")
print(list(df_verify.columns))

Preprocessed dataset successfully saved.


Destination: ..\data\processed\cleaned_dataset.csv
Final Exported Shape: 7308 rows, 13 columns

--- Verified Export Columns ---
['Date', 'Location', 'Rainfall_mm', 'WaterLevel_m', 'SoilMoisture_pct', 'Elevation_m', 'Location_Manila', 'Location_Marikina', 'Location_Pasig', 'Location_Quezon City', 'FloodOccurrence', 'FloodDepth_m', 'RiskLevel']


## Preprocessing Summary & Target Specifications

### 1. Data Cleaning & Integrity
- **Raw File Source:** `data/raw/kaggle_dataset_complete.csv` (7,308 rows, 4 study cities across 2016–2020).
- **Quality Checks:** Zero missing values and zero duplicate records were identified, requiring no row deletion or imputation.
- **Physical Bounds Verification:** Enforced non-negativity checks on all hydrologic fields. Exactly 0 physical anomalies were found.

### 2. Feature Transformations
- **One-Hot Encoding:** Categorical `Location` was converted into binary indicator variables (`Location_Manila`, `Location_Marikina`, `Location_Pasig`, `Location_Quezon City`) to enable standard numeric matrix input for scikit-learn estimators.
- **Multicollinearity Context:** As demonstrated in EDA, `Elevation_m` is perfectly collinear with `Location` dummy variables. In subsequent modeling (Step 3), models will be evaluated with and without `Elevation_m` to prevent multicollinearity distortion in linear baselines.
- **Feature Standardization:** Left out intentionally. `StandardScaler` will be fitted strictly within `03_model_training.ipynb` on `X_train` partitions to prevent data leakage and preserve raw physical units in `cleaned_dataset.csv`.

### 3. Target Variable Definitions & Modeling Constraints
- **Continuous Regression Target (y):** `FloodDepth_m` (in meters), characterized by zero-inflation with peak inundations reaching up to 1.63 m.
- **Multi-Class Classification Target (y):** `RiskLevel`, mapped to the official Mines and Geosciences Bureau (MGB) hazard scale:
  - **Low Susceptibility:** < 0.5 m (7,198 rows, 98.49%)
  - **Moderate Susceptibility:** 0.5 m <= 1.0 m (63 rows, 0.86%)
  - **High Susceptibility:** Flood depth > 1.0 m (47 rows, 0.64%)
- **Data Leakage Safeguards:** 
  - `FloodOccurrence` is retained in the clean dataset strictly for historical benchmarking and auditing; it is completely excluded from feature matrices (X).
  - `FloodDepth_m` and `RiskLevel` are strictly segregated from feature matrices (X) during modeling.
### 4. Output
- Exported clean master dataset to `data/processed/cleaned_dataset.csv`.